# 新闻标题能预测道琼斯指数涨跌吗？

**问题**：每天的世界新闻标题，真的能帮助预测道指（DJIA）第二天/当天的涨跌吗？如果有作用，它在统计上和经济上是真实的吗？

**用户**：一个刷到"AI 读新闻预测股市"广告的普通散户。

This project is a prototype and is not financial advice.

## 阶段 1：环境、数据清理、检查、切分、基准

### 1.0 导入与随机种子
为什么固定随机种子 42：之后的模型、bootstrap 都有随机性，固定种子才能让别人重跑得到一样的数字。

In [1]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# 让 notebook 能 import 项目根目录下的 src/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import data_utils as du
from src.evaluation import evaluate, confusion_table

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA = ROOT / "data"
pd.set_option("display.max_colwidth", 120)

### 1.1 读取并清理新闻数据
- 原始标题很多长这样：`b'Georgia downs two Russian warplanes...'`，这是 Python 字节串被直接写进了 CSV，`b'` 和结尾的 `'` 不是标题内容，要去掉。
- 有少数格子是空的（NaN），换成空字符串，免得后面打情绪分时报错。
- `Date` 转成日期类型并排序，因为之后所有切分都按时间来。

In [2]:
news_raw = pd.read_csv(DATA / "Combined_News_DJIA.csv")
news = du.load_news(DATA / "Combined_News_DJIA.csv")

print("行数、列数:", news.shape)
print("日期范围:", news["Date"].min().date(), "到", news["Date"].max().date())
print("每列标题的空值数（清理前，只显示非零的）:")
na = news_raw[du.HEADLINE_COLS].isna().sum()
print(na[na > 0])

has_prefix = news_raw[du.HEADLINE_COLS].apply(lambda s: s.astype(str).str.match(r'^b[\'"]')).sum().sum()
left = news[du.HEADLINE_COLS].apply(lambda s: s.str.match(r'^b[\'"]')).sum().sum()
print(f"清理前带 b'' 前缀的标题: {has_prefix}，清理后仍带前缀: {left}")

行数、列数: (1989, 27)
日期范围: 2008-08-08 到 2016-07-01
每列标题的空值数（清理前，只显示非零的）:
Top23    1
Top24    3
Top25    3
dtype: int64
清理前带 b'' 前缀的标题: 11920，清理后仍带前缀: 0


In [3]:
# 清理前后对比，看看效果
print("清理前:", news_raw.loc[0, "Top1"])
print("清理后:", news.loc[0, "Top1"])

清理前: b"Georgia 'downs two Russian warplanes' as countries move to brink of war"
清理后: Georgia 'downs two Russian warplanes' as countries move to brink of war


### 1.2 读取道指价格并检查 Label
数据集作者说 Label = 1 表示道指"上涨或持平"。我们用价格自己算一遍，看看和 Label 是否一致。
同时检查两种可能的定义：**收盘价对前一天收盘价**、**收盘价对当天开盘价**。哪种一致率接近 100%，就说明 Label 实际是按哪种定义做的。

In [4]:
# Kaggle 下载下来的文件名是 upload_DJIA_table.csv，两种名字都接受
djia_path = next(p for p in [DATA / "DJIA_table.csv", DATA / "upload_DJIA_table.csv"] if p.exists())
print("价格文件:", djia_path.name)
djia = du.load_djia(djia_path)
print("道指表:", djia.shape, djia["Date"].min().date(), "到", djia["Date"].max().date())

# 两张表的日期是否一一对应
only_news = set(news["Date"]) - set(djia["Date"])
only_djia = set(djia["Date"]) - set(news["Date"])
print("只在新闻表里的日期数:", len(only_news), "| 只在道指表里的日期数:", len(only_djia))

merged, rates = du.check_labels(news, djia)
pd.DataFrame(rates).T

价格文件: upload_DJIA_table.csv
道指表: (1989, 7) 2008-08-08 到 2016-07-01
只在新闻表里的日期数: 0 | 只在道指表里的日期数: 0


,compared_days,agreement
up_adjclose,1988.0,0.998994
up_close,1988.0,0.998994
up_open_close,1989.0,0.983409


In [5]:
# 把不一致的日子列出来看看（用收盘对前一天收盘的定义）
mismatch = merged[merged["up_adjclose"].notna() & (merged["up_adjclose"] != merged["Label"])]
print("不一致的天数:", len(mismatch))
mismatch[["Date", "Label", "Open", "Close", "Adj Close", "up_adjclose", "up_open_close"]].head(15)

不一致的天数: 2


,Date,Label,Open,Close,Adj Close,up_adjclose,up_open_close
1436,2014-04-24,0,16503.390625,16501.650391,16501.650391,1.0,0.0
1839,2015-11-27,0,17806.039062,17813.390625,17813.390625,1.0,1.0


**看不一致那两天的前后几行**：如果收盘价和前一天一模一样，按"涨或持平 = 1"的规则应该是 1，但 Label 是 0。

In [6]:
for d in mismatch["Date"]:
    i = djia.index[djia["Date"] == d][0]
    print(djia.loc[i - 1:i, ["Date", "Open", "Close", "Volume"]].merge(news[["Date", "Label"]], on="Date"), "\n")

        Date          Open         Close    Volume  Label
0 2014-04-23  16513.730469  16501.650391  76830000      0
1 2014-04-24  16503.390625  16501.650391  80340000      0 

        Date          Open         Close    Volume  Label
0 2015-11-25  17820.810547  17813.390625  82540000      1
1 2015-11-27  17806.039062  17813.390625  82540000      0 



**结论**：Label 是按"今天收盘价 ≥ 前一天收盘价"定义的（一致率 99.9%）；"收盘 ≥ 开盘"只有 98.3%，不是它的定义。
（道指是指数，没有分红调整，所以 Close 和 Adj Close 完全一样，两行结果相同。）

仅有的 2 个不一致日（2014-04-24、2015-11-27），价格表里的收盘价和前一天**一模一样**，2015-11-27 连成交量都和前一天相同——很像价格表把前一天的数据复制了一行，是价格表的数据质量问题，而 Label 可能来自另一份价格。
处理方式：**预测目标仍然用原始 Label**（它是题目定义的目标，而且只差 2 天）。但要记住：2015-11-27 在测试集里，阶段 6 用价格算收益时，这一天的收益会被算成 0。

### 1.3 按日期切分
- 训练集：2008-08-08 到 2014-12-31
- 测试集：2015-01-02 到 2016-07-01

为什么按日期而不是随机打乱：现实中我们只能用过去预测未来。如果随机打乱，模型会"偷看"到未来的日子，结果会虚高。

In [7]:
train, test = du.split_by_date(news)
assert train["Date"].max() < test["Date"].min()  # 确认训练集完全在测试集之前
assert len(train) + len(test) == len(news)        # 确认没有日子被漏掉

split_summary = pd.DataFrame({
    "起始": [train["Date"].min().date(), test["Date"].min().date()],
    "结束": [train["Date"].max().date(), test["Date"].max().date()],
    "天数": [len(train), len(test)],
    "涨的比例": [train["Label"].mean(), test["Label"].mean()],
}, index=["训练集", "测试集"])
split_summary

,起始,结束,天数,涨的比例
训练集,2008-08-08,2014-12-31,1611,0.541899
测试集,2015-01-02,2016-07-01,378,0.507937


### 1.4 两个基准（只在测试集上算）
任何 AI 模型都必须先打败这两个"笨办法"，否则就没有价值：
1. **永远预测涨**：股市长期大多数日子在涨，所以这个准确率不会太低。
2. **跟前一天同方向**：检验"动量"——昨天涨今天也涨。测试集第一天用训练集最后一天的 Label。

指标：准确率为主；再看混淆矩阵和"跌"的召回率（真实下跌的日子里，猜中了几成）。永远预测涨的"跌"召回率必然是 0，这说明光看准确率会被骗。

In [8]:
y_test = test["Label"].to_numpy()
preds = {
    "永远预测涨": du.baseline_always_up(test),
    "跟前一天同方向": du.baseline_previous_day(train, test),
}

results = pd.DataFrame([evaluate(y_test, p, name) for name, p in preds.items()]).set_index("model")
results

,accuracy,down_recall
model,,
永远预测涨,0.507937,0.000000
跟前一天同方向,0.457672,0.451613


In [9]:
for name, p in preds.items():
    print(f"\n=== {name} ===")
    print(confusion_table(y_test, p))


=== 永远预测涨 ===
         预测 跌(0)  预测 涨(1)
真实 跌(0)        0      186
真实 涨(1)        0      192

=== 跟前一天同方向 ===
         预测 跌(0)  预测 涨(1)
真实 跌(0)       84      102
真实 涨(1)      103       89


### 阶段 1 小结
- 数据：1989 个交易日，每天 25 条标题；共 11920 条标题带 `b''` 前缀，已全部去掉；只有 7 个空标题（Top23–Top25），已换成空字符串。新闻表和价格表的日期完全对得上。
- 切分：训练集 1611 天（涨 54.2%），测试集 378 天（涨 50.8%）。注意测试期比训练期"更难"：涨的比例低了约 3.4 个百分点，所以训练时学到的"多数时候涨"在测试期没那么好用。
- 基准（测试集）：
  - 永远预测涨：准确率 **50.8%**，但"跌"召回率 **0%**——一次下跌都没抓到。
  - 跟前一天同方向：准确率 **45.8%**，比瞎猜还差；"跌"召回率 45.2%。说明 2015–2016 年道指日涨跌**没有动量**。低于 50% 看起来像"反转"，但 378 天里差 4 个百分点，完全可能是随机波动，不能当成规律。
- 这意味着：之后的新闻模型想说自己有用，至少要稳定超过 **50.8%**；而且因为测试集只有 378 天，几个百分点的差距可能只是运气——这正是阶段 5 要检验的。